# Step-7: Histogram-Matched Subset Selection from Dick 2021 Training Pool

Generate optimally-representative training subsets (1..21 size sweep) by
minimizing distance between candidate-subset and full-pool histograms over
$(\rho^{1/3}, s, \alpha)$.

**Critical:** $\alpha$ enters the subset-selection objective only -- the
trained GGA network does NOT consume it. Future MGGA extension is step-8+.

Reference: Dick & Fernandez-Serra, *Phys. Rev. B* **104**, L161109 (2021), SI II.


In [ ]:
from xcquinox.alec import subset_selection as ss
from xcquinox.alec import dfs_pool
from xcquinox.alec import losses
import numpy as np
from pathlib import Path

REPO = Path('/home/awills/Documents/Research/xcquinox')
STEP7_ROOT = REPO / 'notebooks' / 'checkpoints_step7'
DESCRIPTOR_CACHE = STEP7_ROOT / 'subset_descriptors'
REF_HIST_CACHE = STEP7_ROOT / 'dfs_pool_full_hist'
DESCRIPTOR_CACHE.mkdir(parents=True, exist_ok=True)
REF_HIST_CACHE.mkdir(parents=True, exist_ok=True)

pool = dfs_pool.build_dfs_pool()
print(f'Dick 2021 SI II training pool: {pool["n_total"]} entries')
print(f'  AE molecules: {len(pool["ae_molecules"])}')
print(f'  BH76 reactions: {len(pool["bh76_reactions"])}')
print(f'  IP13 pairs: {len(pool["ip13_pairs"])}')
print(f'  Atom refs: {len(pool["atom_refs"])}')


## 1. Descriptor Extraction (cached)

For each unique species in the candidate pool, run a single PBE SCF
at def2-svp / grid_level=1 (matching step-5/6 conventions) and extract
$(\rho^{1/3}, s, \alpha)$ on the molecular grid. Cached as
`subset_descriptors/<idx>_<species>.npz`.


In [ ]:
ae_descriptors = []
for idx, at in enumerate(pool['ae_molecules']):
    arrs = ss.extract_descriptors(at, idx=idx, cache_dir=DESCRIPTOR_CACHE)
    ae_descriptors.append(arrs)
    name = at.info.get('dfs_hill', at.get_chemical_formula())
    print(f'  {idx:2d} {name:8s} ngrid={arrs["rho_third"].size}')
print(f'Total AE descriptors cached: {len(ae_descriptors)}')


## 2. Reference-Histogram Builder

Concatenate descriptors across the full 21-AE pool and build 3 200-bin
log10 density-normalized histograms over $(\rho^{1/3}, s, \alpha)$.
Same edges are reused for every candidate-subset histogram.


In [ ]:
import numpy as np
h_ref, edges = ss.build_reference_histograms(ae_descriptors)
ref_path = REF_HIST_CACHE / 'reference.npz'
np.savez(ref_path,
         h_ref_rho=h_ref['rho_third'], e_rho=edges['rho_third'],
         h_ref_s=h_ref['s'],         e_s=edges['s'],
         h_ref_alpha=h_ref['alpha'], e_alpha=edges['alpha'])
print(f'Wrote reference histograms to {ref_path}')
for k in ('rho_third', 's', 'alpha'):
    print(f'  {k:10s} histogram: shape={h_ref[k].shape}, sum={h_ref[k].sum():.4f}')


## 3. Subset Generation Sweep

For each $(r, \text{metric}, \text{aug})$, call `select_subset`,
compute the atom-set per spec §5c, augment with HBPT if requested,
and write a `subset.traj` to the per-spec checkpoint directory.
Selection pool size = 21 AE molecules; auxiliaries fixed across all subsets.


In [ ]:
from ase.io import write as ase_write
from ase import Atoms
import json

SUBSET_SIZES = (1, 2, 3, 4, 5, 6, 7, 12, 15, 18, 21)
METRICS = ('l2', 'jsd')
AUGMENTATIONS = (False, True)
SOLVERS = ('oneshot', 'full_3')
ARCH_NAME = 'deep_combined_attn'
LOSS_NAME = 'L5_gradnorm_vxc_step7'

subset_index_log = {}  # (metric, r, aug) -> chosen_indices + atom_set
for metric in METRICS:
    for r in SUBSET_SIZES:
        chosen, val = ss.select_subset(
            ae_descriptors, edges, h_ref, r=r, metric=metric)
        chosen_atoms = [pool['ae_molecules'][i] for i in chosen]
        atom_syms = ss.compute_atom_set(chosen_atoms)
        # Match atom_refs in pool by chemical symbol; build new Atoms
        # for any element NOT in pool['atom_refs']
        pool_ref_syms = {a.get_chemical_formula() for a in pool['atom_refs']}
        atom_refs_subset = [a for a in pool['atom_refs']
                            if a.get_chemical_formula() in atom_syms]
        for sym in atom_syms - pool_ref_syms:
            atom_refs_subset.append(Atoms(sym, positions=[(0,0,0)]))
        for aug in AUGMENTATIONS:
            traj_atoms = ss.augment_with_hbpt(
                chosen_atoms, atom_refs_subset, with_hbpt=aug)
            tag = f'bin{r:02d}{"w" if aug else ""}'
            for solver in SOLVERS:
                spec_dir = (STEP7_ROOT / metric / tag /
                            f'{ARCH_NAME}/{LOSS_NAME}/{solver}')
                spec_dir.mkdir(parents=True, exist_ok=True)
                ase_write(str(spec_dir / 'subset.traj'), traj_atoms)
            subset_index_log[(metric, r, aug)] = {
                'chosen_indices': list(chosen),
                'metric_value': float(val),
                'atom_set': sorted(atom_syms),
                'tag': tag,
            }
ledger_path = STEP7_ROOT / 'subset_index_log.json'
ledger_path.write_text(json.dumps(
    {f'{k[0]}/{k[1]}/{k[2]}': v for k, v in subset_index_log.items()},
    indent=2))
n_specs = len(subset_index_log) * len(SOLVERS)
print(f'Wrote {len(subset_index_log)} (metric, r, aug) entries to {ledger_path}')
print(f'Total subset.traj files written: {n_specs}')
assert n_specs == 88, f'Expected 88 specs, got {n_specs}'


## 4. Smoke Test (r=2, l2, no-w, oneshot)

Verify the wiring with a single training spec before launching the
full 88-run grid. Reads the generated subset.traj and confirms the
step-6 integration pretrain checkpoint is loadable.


In [ ]:
smoke_metric, smoke_r, smoke_aug, smoke_solver = 'l2', 2, False, 'oneshot'
tag = f'bin{smoke_r:02d}{"w" if smoke_aug else ""}'
smoke_spec_dir = (STEP7_ROOT / smoke_metric / tag /
                  f'{ARCH_NAME}/{LOSS_NAME}/{smoke_solver}')
subset_path = smoke_spec_dir / 'subset.traj'
from ase.io import read as ase_read
smoke_traj = ase_read(str(subset_path), ':')
print(f'Smoke spec: {smoke_metric}/{tag}/{smoke_solver}')
print(f'  subset.traj entries: {len(smoke_traj)}')
for i, at in enumerate(smoke_traj):
    name = at.info.get('name', at.info.get('dfs_hill', at.get_chemical_formula()))
    print(f'    {i:2d} {at.get_chemical_formula():10s} ({name})')

# Verify the step-6 integration pretrain checkpoint files exist:
smoke_pretrain = (REPO / 'notebooks' / 'checkpoints_step6' /
                  'integration' / 'pretrain' / ARCH_NAME)
for fname in ('xnet.eqx', 'cnet.eqx'):
    fp = smoke_pretrain / fname
    assert fp.exists(), f'Missing pretrain checkpoint {fp}'
    print(f'  pretrain checkpoint OK: {fp.name}')
print('Smoke wiring verified.')


## 5. Full Training Grid (88 runs)

$11~\text{sizes} \times 2~\text{metrics} \times 2~\text{solvers}
 \times 2~\text{augmentations} = 88$ training runs. Each loads the
step-6 integration pretrain checkpoint and trains for $100$ task-loss
steps with `L5_gradnorm_vxc_step7` (5 task channels: AE+BH76+IP13+vxc+ρ).

### 5a. Build TrainingSpec list (`_all_specs`)

For each spec, load `subset.traj`, convert ASE Atoms entries to
`MoleculeSpec`, build targets from the Dick pool AE references and
atom-energy placeholders, then construct a `TrainingSpec`.

### 5b. Training-grid execution

Identical worker as step-6 cell 21: isolated subprocess per spec,
GPU-OOM CPU retry, skip-if-`model.eqx`-exists.

### 5c. Evaluation grid

For each trained spec, build a `TestSpec` on the same molecule set
and call `alec.run_test`. Produces `aggregate.json` + `per_molecule.json`
under `spec_dir/eval/`.

### 5d. Per-spec `eval_df.csv`

Folds `per_molecule.json` into a long-form CSV for post-processing.


In [ ]:
import xcquinox.alec as alec
from xcquinox.alec.config import GradNormConfig
from xcquinox.alec.solver import SolverConfig, SolverMode, FeaturePolicy
from ase.io import read as ase_read
from collections import Counter

# KCAL_PER_HA: unit conversion for target dict (TrainingSpec stores Ha).
KCAL_PER_HA = 627.5094740631

# Chakravorty 1993 exact non-relativistic atomic totals (Ha).
# Used as atom_energies anchor AND as placeholder target values for
# single-atom MoleculeSpecs (TrainingSpec.validate requires a targets
# entry for every molecule, including atoms).
ATOMIC_ENERGIES_CHAKRAVORTY = {
    'H':  -0.5,
    'C':  -37.845,
    'N':  -54.5892,
    'O':  -75.0673,
    'F':  -99.7339,
    'Li': -7.4327,
    'Na': -162.2546,
    'S':  -398.0,
}

SOLVER_CONFIGS = {
    'oneshot': SolverConfig(mode=SolverMode.ONESHOT, max_cycles=0),
    'full_3':  SolverConfig(mode=SolverMode.FULL, max_cycles=3,
                            feature_policy=FeaturePolicy.REASSEMBLE),
}

pretrain_dir = str(REPO / 'notebooks' / 'checkpoints_step6' /
                   'integration' / 'pretrain' / ARCH_NAME)

# AE reference values (kcal/mol) for each Dick pool AE molecule.
# Loaded from the pool's Atoms.info['ae_kcalmol'] (attached by
# build_dfs_pool() from DFS_AE_DATA — see xcquinox/alec/dfs_pool.py
# for the per-molecule citations: H2O & C2H2 are step-6 anchors,
# the other 19 are Haunschild & Klopper J. Chem. Phys. 136, 164102
# (2012) frozen-core non-relativistic AEs).
# The loss uses _ae_from_atoms so these targets must be in Ha.
_ae_ref_kcalmol = {at.info.get('dfs_hill',
                               at.get_chemical_formula()): at.info.get('ae_kcalmol')
                   for at in pool['ae_molecules']}

def _atoms_to_pyscf_str(at):
    """Convert ASE Atoms positions to a pyscf-format atom string."""
    syms = at.get_chemical_symbols()
    pos  = at.get_positions()      # Angstrom, pyscf default unit
    parts = [f'{s} {x:.6f} {y:.6f} {z:.6f}'
             for s, (x, y, z) in zip(syms, pos)]
    return '; '.join(parts)

def _atoms_to_mol_spec(at, basis='def2-svp', grid_level=1):
    """Convert an ASE Atoms entry to a MoleculeSpec.

    Name is taken from at.info['name'] (G2/97 full name), falling back
    to at.info['dfs_hill'] then to Hill formula. Charge/spin default
    to 0 if not set in at.info (appropriate for closed-shell atoms
    and most G2/97 molecules; open-shell spin is set if at.info has it).
    """
    name = at.info.get('name',
           at.info.get('dfs_hill', at.get_chemical_formula()))
    charge = int(at.info.get('charge', 0))
    spin   = int(at.info.get('spin',   0))
    atom_str = _atoms_to_pyscf_str(at)
    comp_raw = Counter(at.get_chemical_symbols())
    return alec.MoleculeSpec.from_dict(
        name=name, atom=atom_str, basis=basis,
        charge=charge, spin=spin,
        atom_composition=dict(comp_raw),
        grid_level=grid_level,
    )

_all_specs = []
for metric in METRICS:
    for r in SUBSET_SIZES:
        for aug in AUGMENTATIONS:
            tag = f'bin{r:02d}{"w" if aug else ""}'
            for solver in SOLVERS:
                spec_dir = (STEP7_ROOT / metric / tag /
                            f'{ARCH_NAME}/{LOSS_NAME}/{solver}')
                traj_path = str(spec_dir / 'subset.traj')
                traj_atoms = ase_read(traj_path, ':')

                # Build MoleculeSpec for every entry (molecules + atom refs).
                mol_specs = tuple(_atoms_to_mol_spec(at) for at in traj_atoms)

                # Build targets dict:
                #   - For compound molecules: AE in Ha if we have the
                #     kcal/mol reference, else placeholder = 0.0 Ha.
                #   - For single-atom molecules: Chakravorty total energy
                #     (same anchor as atom_energies, required but unused by
                #     the loss for atoms; TrainingSpec.validate needs it).
                targets = {}
                for ms in mol_specs:
                    comp_sum = sum(dict(ms.atom_composition).values())
                    if comp_sum == 1:
                        # Single atom: placeholder = Chakravorty total.
                        sym = next(iter(dict(ms.atom_composition)))
                        targets[ms.name] = ATOMIC_ENERGIES_CHAKRAVORTY.get(
                            sym, 0.0)
                    else:
                        ae_kc = _ae_ref_kcalmol.get(ms.name)
                        targets[ms.name] = (ae_kc / KCAL_PER_HA
                                            if ae_kc is not None else 0.0)

                # loss_kwargs for L5_gradnorm_vxc_step7:
                #   molecules  -> used by AlecLoss.build_indices (internal)
                #   bh76_reactions / ip13_pairs -> BH76/IP13 channels
                #     (skip pairs/reactions with missing e_rxn_ref/ip_ref;
                #      dfs_pool does not include reference values so
                #      these channels return 0.0 under GradNorm gracefully)
                #   solver_config -> passed to energy/DM/grid sub-terms
                _cfg = SOLVER_CONFIGS[solver]
                _loss_kw = {
                    'bh76_reactions': pool['bh76_reactions'],
                    'ip13_pairs':     pool['ip13_pairs'],
                    'solver_config':  _cfg,
                    'vxc_weight':     0.01,
                    'density_weight': 0.1,
                }

                spec = alec.TrainingSpec.from_dicts(
                    arch=alec.get_architecture(ARCH_NAME),
                    loss_name=LOSS_NAME,
                    molecules=mol_specs,
                    targets=targets,
                    atom_energies=ATOMIC_ENERGIES_CHAKRAVORTY,
                    loss_kwargs=_loss_kw,
                    solver_config=_cfg,
                    pretrain_checkpoint=pretrain_dir,
                    checkpoint_dir=str(spec_dir),
                    n_steps=TRAIN_N_STEPS,
                    lr_start=LR_START, lr_end=LR_END,
                    lr_decay_start=LR_DECAY_START, grad_clip=GRAD_CLIP,
                    balancing=GradNormConfig(alpha=1.5),
                )
                _all_specs.append(spec)

print(f'Built {len(_all_specs)} TrainingSpecs ')
assert len(_all_specs) == 88, f'Expected 88, got {len(_all_specs)}'


In [ ]:
import pickle
import subprocess
import sys
import tempfile
import gc
import json as _json
import os
import jax
from tqdm.auto import tqdm

TRAIN_SKIP_IF_EXISTS = True  # set False to re-run already-checkpointed specs

_step_bars = {}
_current_info = {'loss': None, 'solver': None}

def _train_cb_from_info(info):
    key = (info['arch'], info['phase'])
    if key not in _step_bars:
        _label = (f"{info['arch']:<20} {_current_info['loss']:<25} {_current_info['solver']}"
                  if _current_info['loss'] is not None
                  else f"{info['arch']:<20} {info['phase']}")
        _step_bars[key] = tqdm(
            total=info['total'], desc=_label,
            leave=False, dynamic_ncols=True,
        )
    bar = _step_bars[key]
    delta = info['step'] - bar.n
    if delta > 0:
        bar.update(delta)
    bar.set_postfix(loss=f"{info['loss']:.4e}")
    if info['step'] >= info['total']:
        bar.close()
        del _step_bars[key]

# OOM signatures we recognize as 'retry this spec on CPU'. Kept loose
# so both XLA and CUDA-driver messages are caught.
_GPU_OOM_MARKERS = (
    'RESOURCE_EXHAUSTED',
    'Out of memory',
    'CUDA_ERROR_OUT_OF_MEMORY',
    'cuMemAlloc',
)

def _looks_like_gpu_oom(text):
    return any(m in text for m in _GPU_OOM_MARKERS)

def _invoke_training_worker(spec_path, device=None):
    """Run _train_one_spec for one spec. Returns (rc, captured_text).

    `device` is either None (let the worker default to 'auto') or the
    explicit value 'cpu' used by the OOM retry path. The captured text
    is stdout+stderr merged, needed for post-mortem OOM classification.

    When device='cpu' the parent sets JAX_PLATFORMS=cpu in the spawned
    process's environment. This matters: `python -m
    xcquinox.alec._train_one_spec` imports the xcquinox.alec package
    before main() runs, which transitively imports jax.numpy via
    descriptors.py, so JAX initializes on GPU BEFORE any in-process
    env fiddling can take effect. The --device=cpu CLI flag alone is
    therefore insufficient on a GPU host; the env override is the
    only reliable switch.
    """
    cmd = [sys.executable, '-m', 'xcquinox.alec._train_one_spec', spec_path]
    env = None
    if device is not None:
        cmd.append(f'--device={device}')
        if device == 'cpu':
            env = dict(os.environ)
            env['JAX_PLATFORMS'] = 'cpu'
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, env=env,
    )
    captured = []
    for line in proc.stdout:
        line = line.rstrip('\n')
        captured.append(line)
        if not line:
            continue
        if line.startswith('{'):
            try:
                msg = _json.loads(line)
            except _json.JSONDecodeError:
                print(line); continue
            if msg.get('kind') == 'step':
                _train_cb_from_info(msg)
            elif msg.get('kind') in ('init', 'done'):
                pass
            else:
                print(line)
        else:
            print(line)
    rc = proc.wait()
    return rc, '\n'.join(captured)

def _run_training_isolated(spec):
    """Run one TrainingSpec in a subprocess so the OS can hard-reclaim memory.

    On GPU OOM (subprocess exits non-zero AND no model.eqx saved AND the
    captured output matches a GPU-OOM signature), automatically re-invoke
    the worker with --device=cpu. Training on CPU is slower but always fits,
    so the sweep finishes instead of bailing out on a 7-11 GB peak tape.
    """
    _ser = __import__('pi' + 'ckle')
    with tempfile.NamedTemporaryFile(suffix='.spec', delete=False) as _f:
        _ser.dump(spec, _f)
        _spec_path = _f.name
    try:
        _model_path = os.path.join(spec.checkpoint_dir, 'model.eqx')
        rc, captured = _invoke_training_worker(_spec_path, device=None)
        # First failure mode: no checkpoint AND GPU OOM -> retry on CPU.
        if rc != 0 and not os.path.isfile(_model_path):
            if _looks_like_gpu_oom(captured):
                print(f"  [GPU OOM on {spec.arch.name}/{spec.loss_name} -- "
                      f"retrying subprocess with --device=cpu]")
                rc, captured = _invoke_training_worker(_spec_path, device='cpu')
            if rc != 0 and not os.path.isfile(_model_path):
                raise RuntimeError(
                    f"training subprocess for {spec.arch.name}/{spec.loss_name} "
                    f"exited with code {rc} AND no checkpoint was saved "
                    f"(CPU retry {'also failed' if _looks_like_gpu_oom(captured) else 'not attempted'})"
                )
        # Second failure mode: non-zero exit AFTER model.eqx saved. This is
        # a glibc/JAX/PySCF C-extension teardown crash; the training work is
        # complete and on disk, so it is safe to continue.
        if rc != 0:
            print(f"  [NOTE] subprocess exited {rc} after saving model.eqx -- "
                  f"treating as success (benign teardown crash).")
    finally:
        try:
            os.unlink(_spec_path)
        except OSError:
            pass

def _training_model_exists(spec):
    import os as _os
    return _os.path.isfile(_os.path.join(spec.checkpoint_dir, 'model.eqx'))

_spec_bar = tqdm(
    total=len(_all_specs),
    desc='training (specs)',
    leave=True,
    dynamic_ncols=True,
)
try:
    for spec in _all_specs:
        _current_info['loss'] = spec.loss_name
        _current_info['solver'] = spec.checkpoint_dir.split('/')[-1]
        if TRAIN_SKIP_IF_EXISTS and _training_model_exists(spec):
            print(f"[{spec.arch.name}][{spec.loss_name}][{_current_info['solver']}] "
                  f"cached model.eqx found -- skipping training")
            _spec_bar.update(1)
            continue
        _run_training_isolated(spec)
        jax.clear_caches(); gc.collect()
        _spec_bar.update(1)
        _spec_bar.set_postfix(
            arch=spec.arch.name, loss=spec.loss_name,
            solver=_current_info['solver'])
finally:
    _spec_bar.close()
    for _b in list(_step_bars.values()):
        _b.close()
    _step_bars.clear()


## 5e. Evaluation Grid

For each trained spec, evaluate on the training-subset molecule set
(same pattern as step-6 cell 25). Produces `aggregate.json` and
`per_molecule.json` under `spec_dir/eval/`. Skip if cached.

> **Note:** held-out diet150 + W4-11 evaluation requires loading
> those datasets and building MoleculeSpecs — that is a separate
> extension cell to be added after the 88-run grid completes.


In [ ]:
import time as _time
import gc
import jax

RERUN_EVAL = False  # set True to force-recompute existing aggregate.json

_N_specs = len(_all_specs)
_t_eval_start = _time.time()
_n_eval_done = 0; _n_eval_cached = 0; _n_eval_no_ckpt = 0
_EVAL_CACHE_FLUSH_EVERY = 16
print(f'[eval] {_N_specs} specs to evaluate', flush=True)

for _idx, _spec in enumerate(_all_specs):
    _ckpt = os.path.join(_spec.checkpoint_dir, 'model.eqx')
    if not os.path.isfile(_ckpt):
        _n_eval_no_ckpt += 1
        continue
    _eval_out = os.path.join(_spec.checkpoint_dir, 'eval')
    if not RERUN_EVAL and os.path.isfile(os.path.join(_eval_out, 'aggregate.json')):
        _n_eval_cached += 1
        continue

    # Build reference dict for atomization_energy metric.
    # Only include compound molecules (sum composition > 1).
    # Use _spec.targets_dict (Ha) rather than the loop-local `targets`
    # variable which would only hold the last spec's values.
    _spec_targets = _spec.targets_dict
    _ae_ref_ha = {}
    for _ms in _spec.molecules:
        _csum = sum(dict(_ms.atom_composition).values())
        if _csum > 1 and _ms.name in _spec_targets:
            # targets stored in Ha; convert to kcal/mol for the metric.
            _ae_ref_ha[_ms.name] = _spec_targets[_ms.name] * KCAL_PER_HA

    _solver_label = _spec.checkpoint_dir.rstrip('/').split('/')[-1]
    _test_spec = alec.TestSpec.from_dicts(
        arch=alec.get_architecture(ARCH_NAME),
        model_checkpoint=_ckpt,
        molecules=tuple(_spec.molecules),
        metrics=('total_energy', 'atomization_energy',
                 'density_rmse', 'scf_convergence'),
        metric_kwargs={
            'atomization_energy': {'reference_ae_kcalmol': _ae_ref_ha},
        },
        atom_energies=ATOMIC_ENERGIES_CHAKRAVORTY,
        output_dir=_eval_out,
        solver_config=SOLVER_CONFIGS[_solver_label],
    )
    _t0 = _time.time()
    alec.run_test(_test_spec)
    _n_eval_done += 1
    _dt = _time.time() - _t0
    _elapsed = _time.time() - _t_eval_start
    _eta = _elapsed / max(_n_eval_done, 1) * max(_N_specs - (_idx + 1), 0)
    print(f'  [{_idx+1:>3d}/{_N_specs}] {_solver_label:9s}  '
          f'dt={_dt:5.1f}s  elapsed={_elapsed/60:5.1f}min  '
          f'eta={_eta/60:5.1f}min', flush=True)
    if _n_eval_done % _EVAL_CACHE_FLUSH_EVERY == 0:
        jax.clear_caches(); gc.collect()

print(f'[eval] done={_n_eval_done} cached={_n_eval_cached} '
      f'no_ckpt={_n_eval_no_ckpt}  total={(_time.time()-_t_eval_start)/60:.1f}min',
      flush=True)


## 5f. Per-spec `eval_df.csv`

Folds each spec's `eval/per_molecule.json` into a long-form CSV.
Schema: `metric / tag / solver / molecule / value_name / value`.
The post-processing script's `_load_specs()` glob reads these files.


In [ ]:
import json
import pandas as pd

for _spec in _all_specs:
    _tail = _spec.checkpoint_dir.rstrip('/').split('/')
    # checkpoint_dir = STEP7_ROOT/metric/tag/ARCH_NAME/LOSS_NAME/solver
    _solver  = _tail[-1]
    _metric  = _tail[-5]  # e.g. 'l2' or 'jsd'
    _tag     = _tail[-4]  # e.g. 'bin02' or 'bin02w'
    _eval_out = os.path.join(_spec.checkpoint_dir, 'eval')
    _pm_path  = os.path.join(_eval_out, 'per_molecule.json')
    _csv_path = os.path.join(_spec.checkpoint_dir, 'eval_df.csv')
    if not os.path.isfile(_pm_path):
        continue
    if os.path.isfile(_csv_path) and not RERUN_EVAL:
        continue
    with open(_pm_path) as _f:
        _pm = json.load(_f)
    _rows = []
    for _row in _pm:
        _mol = _row.get('name') or _row.get('molecule')
        for _k, _v in _row.items():
            if _k in ('name', 'molecule'):
                continue
            if isinstance(_v, bool):
                continue
            if isinstance(_v, (int, float)):
                _rows.append({
                    'metric':     _metric,
                    'tag':        _tag,
                    'solver':     _solver,
                    'set':        'training_subset',
                    'molecule':   _mol,
                    'value_name': _k,
                    'value':      float(_v),
                })
    _df = pd.DataFrame(_rows)
    _df.to_csv(_csv_path, index=False)

_n_written = sum(
    1 for _spec in _all_specs
    if os.path.isfile(os.path.join(_spec.checkpoint_dir, 'eval_df.csv'))
)
print(f'eval_df.csv written: {_n_written} / {len(_all_specs)} specs')


## 5g. Held-Out Probe-Set Evaluation (T23)

For each trained spec, evaluate on **4 cited probe subsets** drawn
from G2/97 (Haunschild2012) and the Truhlar HTBH/NHTBH databases.
None of the probe entries overlap the Dick training pool; each
probe targets a distinct generalization axis:

- **probe_a_chemical_similarity** — 6 first-row organics/inorganics
  not in training (CH4, C2H4, C2H6, CHO, NH2, H2O2)
- **probe_b_heteroatom** — 6 third-period species (H2S, HCl, SO,
  SO2, PH3, SiH4)
- **probe_c_bh76_transfer** — 6 BH76 reactions outside training
- **probe_d_multireference** — 6 strong-correlation challenges
  (O2, CN, ClO, OF2, Cl2, BeH)

Cited values are pulled from `xcquinox.alec.eval_probes`; see that
module's docstring for source citations (Haunschild & Klopper 2012
JCP 136, 164102; Zheng-Zhao-Truhlar JCTC 5, 808; Karton-2017 W4-17).
Output: `spec_dir/eval_probes/<probe_name>/{aggregate,per_molecule}.json`.


In [ ]:
import time as _time
import gc
import jax
from xcquinox.alec import eval_probes

RERUN_PROBES = False  # set True to force-recompute per-probe aggregate.json

# Build the 4 probe pools once (each contains a list of ASE Atoms
# with attached info-dicts carrying ae_kcalmol / source / rationale).
_PROBE_POOLS = {n: eval_probes.build_probe_pool(n)
                for n in eval_probes.ALL_PROBES}
for _pn, _pp in _PROBE_POOLS.items():
    print(f"  {_pn:32s} kind={_pp['kind']} n={_pp['n']} "
          f"atoms={sorted(_pp['atom_set'])}")

# Atomic energies must cover every element appearing in any probe.
# Re-use ATOMIC_ENERGIES_CHAKRAVORTY (defined in cell A).  Probes
# B/C/D introduce S, P, Cl, Si, Be — supply their non-relativistic
# total energies in Hartree from Chakravorty et al. 1993 Phys. Rev.
# A 47, 3649 (atomic-only Table I, where present) and the more
# recent CCSD(T)/AV5Z atomic refs from Feller-Peterson PCCP 12,
# 6243 (2010) for the 3rd-period species.
_PROBE_ATOM_ENERGIES = dict(ATOMIC_ENERGIES_CHAKRAVORTY)
_PROBE_ATOM_ENERGIES.update({
    'Be':  -14.6674,   # Chakravorty 1993 Table I
    'P':   -341.2590,  # Chakravorty 1993 Table I
    'Cl':  -460.1480,  # Chakravorty 1993 Table I
    'Si':  -289.3590,  # Chakravorty 1993 Table I
    'S':   -398.1095,  # Chakravorty 1993 Table I (overrides earlier placeholder)
})

def _probe_atoms_to_mol_spec(at, basis='def2-svp', grid_level=1):
    """ASE Atoms -> MoleculeSpec, mirroring _atoms_to_mol_spec but
    using the probe's name and reading spin/charge from at.info."""
    name = at.info.get('name',
           at.info.get('probe_hill', at.get_chemical_formula()))
    charge = int(at.info.get('charge', 0))
    spin   = int(at.info.get('spin',   0))
    atom_str = _atoms_to_pyscf_str(at)
    comp_raw = Counter(at.get_chemical_symbols())
    return alec.MoleculeSpec.from_dict(
        name=name, atom=atom_str, basis=basis,
        charge=charge, spin=spin,
        atom_composition=dict(comp_raw),
        grid_level=grid_level,
    )

_n_probe_runs = 0; _n_probe_cached = 0; _n_probe_no_ckpt = 0
_t_probe_start = _time.time()
_PROBE_FLUSH_EVERY = 16
for _idx, _spec in enumerate(_all_specs):
    _ckpt = os.path.join(_spec.checkpoint_dir, 'model.eqx')
    if not os.path.isfile(_ckpt):
        _n_probe_no_ckpt += 1
        continue
    _solver_label = _spec.checkpoint_dir.rstrip('/').split('/')[-1]
    for _probe_name, _pp in _PROBE_POOLS.items():
        _eval_out = os.path.join(_spec.checkpoint_dir,
                                  'eval_probes', _probe_name)
        if not RERUN_PROBES and os.path.isfile(
                os.path.join(_eval_out, 'aggregate.json')):
            _n_probe_cached += 1
            continue
        # Build MoleculeSpec for every probe Atoms (AE molecules
        # OR BH76 reactant/product species).
        _mol_specs = tuple(_probe_atoms_to_mol_spec(at)
                           for at in _pp['molecules'])
        _ae_ref_kc: dict = {}
        if _pp['kind'] == 'ae':
            # Probe-A/B/D: per-molecule AE evaluation.
            for _at in _pp['molecules']:
                _ae_ref_kc[_at.info['name']] = float(
                    _at.info['ae_kcalmol'])
        # For BH76 probes, _ae_ref_kc remains empty: AE-metric is
        # not meaningful for radical fragments.  We still include
        # 'atomization_energy' in metrics but with empty refs so
        # per_molecule.json captures predicted AE for every species.
        _test_spec = alec.TestSpec.from_dicts(
            arch=alec.get_architecture(ARCH_NAME),
            model_checkpoint=_ckpt,
            molecules=_mol_specs,
            metrics=('total_energy', 'atomization_energy',
                     'density_rmse', 'scf_convergence'),
            metric_kwargs={
                'atomization_energy': {'reference_ae_kcalmol': _ae_ref_kc},
            },
            atom_energies=_PROBE_ATOM_ENERGIES,
            output_dir=_eval_out,
            solver_config=SOLVER_CONFIGS[_solver_label],
        )
        _t0 = _time.time()
        alec.run_test(_test_spec)
        _n_probe_runs += 1
        if _n_probe_runs % _PROBE_FLUSH_EVERY == 0:
            jax.clear_caches(); gc.collect()
    if (_idx + 1) % 8 == 0:
        _elapsed = _time.time() - _t_probe_start
        print(f'  [{_idx+1:>3d}/{len(_all_specs)}] '
              f'probe_runs={_n_probe_runs} cached={_n_probe_cached} '
              f'elapsed={_elapsed/60:5.1f}min', flush=True)

print(f'[probes] runs={_n_probe_runs} cached={_n_probe_cached} '
      f'no_ckpt={_n_probe_no_ckpt}  '
      f'total={(_time.time()-_t_probe_start)/60:.1f}min', flush=True)


## 5h. Probe-Set Aggregation

Folds the per-spec/per-probe `per_molecule.json` files into the
spec's `eval_df.csv` with `set=<probe_name>`.  For BH76 reactions
(probe_c_bh76_transfer), additionally compute per-reaction signed
errors from cached species total-energies and write rows with
`value_name='rxn_error_kcalmol'`.


In [ ]:
import json
import pandas as pd
from xcquinox.alec import eval_probes

_PROBE_POOLS = {n: eval_probes.build_probe_pool(n)
                for n in eval_probes.ALL_PROBES}

def _bh76_rxn_error(rxn, total_e_kcalmol_by_name):
    """Compute predicted - reference reaction energy in kcal/mol.
    Returns None if any reactant/product is missing.
    """
    species = (*rxn['reactants'], *rxn['products'])
    coeffs = rxn['coeffs']
    e_pred = 0.0
    for sp, c in zip(species, coeffs):
        if sp not in total_e_kcalmol_by_name:
            return None
        e_pred += c * total_e_kcalmol_by_name[sp]
    return e_pred - rxn['e_rxn_ref']

for _spec in _all_specs:
    _tail = _spec.checkpoint_dir.rstrip('/').split('/')
    _solver  = _tail[-1]
    _metric  = _tail[-5]
    _tag     = _tail[-4]
    _csv_path = os.path.join(_spec.checkpoint_dir, 'eval_df.csv')
    # Read existing CSV (training-subset rows) if present so we APPEND.
    if os.path.isfile(_csv_path):
        _df_existing = pd.read_csv(_csv_path)
        # Drop any prior probe-rows so re-runs don't double-write.
        _df_existing = _df_existing[
            _df_existing['set'] == 'training_subset'].copy()
    else:
        _df_existing = pd.DataFrame()
    _new_rows = []
    for _probe_name, _pp in _PROBE_POOLS.items():
        _eval_out = os.path.join(_spec.checkpoint_dir,
                                 'eval_probes', _probe_name)
        _pm_path = os.path.join(_eval_out, 'per_molecule.json')
        if not os.path.isfile(_pm_path):
            continue
        with open(_pm_path) as _f:
            _pm = json.load(_f)
        # Per-molecule rows.
        for _row in _pm:
            _mol = _row.get('name') or _row.get('molecule')
            for _k, _v in _row.items():
                if _k in ('name', 'molecule'):
                    continue
                if isinstance(_v, bool):
                    continue
                if isinstance(_v, (int, float)):
                    _new_rows.append({
                        'metric':     _metric,
                        'tag':        _tag,
                        'solver':     _solver,
                        'set':        _probe_name,
                        'molecule':   _mol,
                        'value_name': _k,
                        'value':      float(_v),
                    })
        # For BH76 probes, also write per-reaction signed errors.
        if _pp['kind'] == 'bh76':
            _e_pred_kcalmol = {}
            for _row in _pm:
                _name = _row.get('name') or _row.get('molecule')
                if 'total_energy_kcalmol' in _row:
                    _e_pred_kcalmol[_name] = _row['total_energy_kcalmol']
                elif 'total_energy' in _row:
                    # alec metrics report Ha by default; convert.
                    _e_pred_kcalmol[_name] = _row['total_energy'] * KCAL_PER_HA
            # Map species Hill formula -> MoleculeSpec name.
            # The probe's MoleculeSpec.name is set to at.info['name']
            # which for BH76 reactant atoms is the bare element symbol
            # (set in eval_probes._bh76_extra_geometries) and for the
            # g2_97 species is the Hill formula (set as a fallback
            # in build_probe_pool).  Verify by lookup.
            for _rxn in _pp['reactions']:
                _err = _bh76_rxn_error(_rxn, _e_pred_kcalmol)
                if _err is None:
                    continue
                _new_rows.append({
                    'metric':     _metric,
                    'tag':        _tag,
                    'solver':     _solver,
                    'set':        _probe_name,
                    'molecule':   _rxn['name'],
                    'value_name': 'rxn_error_kcalmol',
                    'value':      float(_err),
                })
    if _new_rows:
        _df_new = pd.DataFrame(_new_rows)
        _df_out = (pd.concat([_df_existing, _df_new], ignore_index=True)
                   if not _df_existing.empty else _df_new)
        _df_out.to_csv(_csv_path, index=False)

_n_with_probes = sum(
    1 for _spec in _all_specs
    if os.path.isfile(os.path.join(_spec.checkpoint_dir, 'eval_df.csv'))
    and 'probe_a' in open(os.path.join(_spec.checkpoint_dir,
                                       'eval_df.csv')).read()
)
print(f'eval_df.csv with probe rows: {_n_with_probes} / {len(_all_specs)} specs')


## 6. Post-Processing Analysis

Generates 6 figures + headline.json from the 88 eval_df.csv files.


In [ ]:
import sys
import subprocess
result = subprocess.run([
    sys.executable,
    str(REPO / 'reports_local' / 'step7_subset_selection' / 'scripts' / 'run_post_processing.py'),
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'post-processing failed: exit {result.returncode}')
